# 🛡️ EduGuard: Proactive Identification of At-Risk Students Using Machine Learning

**Author:** Hira Sajjad

Can we prevent students from dropping out before it happens?

This project uses the Open University Learning Analytics Dataset (OULAD) to build an interpretable machine learning model that proactively flags students at risk of withdrawing. By leveraging Explainable AI (SHAP), EduGuard goes beyond simple predictions. It provides educators with the exact reasons a student is struggling, enabling targeted and effective interventions.

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score

print("✅ All libraries imported successfully!")

In [ ]:
# Load the data
df = pd.read_csv('final_dataset.csv')

print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}\n")

print("--- First 5 Rows ---")
display(df.head())

print("\n--- Target Variable Distribution ---")
print(df['final_result'].value_counts())

In [ ]:
# 1. Create Binary Target (1 = At-Risk, 0 = Safe)
df['dropout'] = df['final_result'].apply(lambda x: 1 if x in ['Withdrawn', 'Fail'] else 0)

print("--- New Target Distribution ---")
print(df['dropout'].value_counts())
print(f"\nDropout Rate: {df['dropout'].mean():.2%}")

# 2. Separate Features (X) and Target (y)
X = df.drop(['id_student', 'final_result', 'dropout'], axis=1)
y = df['dropout']

# 3. One-Hot Encode Categorical Variables
X = pd.get_dummies(X, drop_first=True)

print(f"\nFinal Feature Shape: {X.shape}")
print("✅ Data is ready for Machine Learning!")

In [ ]:
# 1. Split into Training and Testing sets (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 2. Train the Model
print("Training the Random Forest model... (please wait)")
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# 3. Evaluate the Model
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]

print("\n✅ Model Trained!")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_prob):.4f}")
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Safe', 'At-Risk']))

## 🧠 Explainable AI (SHAP) Insights

Our Random Forest model achieved an outstanding 93.5% accuracy and 0.98 ROC-AUC in predicting student dropout. But accuracy alone isn't enough. Educators need to know exactly why a student is at risk.

Using SHAP (SHapley Additive exPlanations), we can open the black box of our AI:

1. **Global Feature Importance (Bar Chart):** Reveals the top factors driving dropout across all students.
   * Key Insight: Previous academic attempts and current study load are the strongest predictors of attrition.
2. **Individual Force Plot:** Explains the exact reasoning behind a single student's prediction.
   * Key Insight: Red bars push a student toward being At-Risk, while blue bars pull them toward being Safe. This allows teachers to intervene with targeted support.

In [ ]:
import shap
import matplotlib.pyplot as plt

# --- 1. Create the SHAP Explainer ---
print("Calculating SHAP values... (this takes a few seconds, please wait)")
explainer = shap.TreeExplainer(rf_model)

# --- 2. Compute SHAP values ---
# In newer versions, this returns a 3D array: (n_samples, n_features, n_classes)
shap_values = explainer.shap_values(X_test)

# --- 3. The Fix: Select the SHAP values for the 'At-Risk' class (index 1) ---
shap_values_at_risk = shap_values[:, :, 1]  # This extracts the 2D matrix for class 1

# --- 4. Global Feature Importance Plot ---
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_at_risk, X_test, plot_type="bar", show=False)
plt.title("EduGuard: Top Factors Driving Student Dropout", fontsize=14)
plt.tight_layout()
plt.show()

print("✅ SHAP summary plot generated!")

In [ ]:
student_idx = 0
print(f"Generating Force Plot for Student #{student_idx}...\n")

shap.force_plot(
    explainer.expected_value[1],
    shap_values[student_idx, :, 1], # Correct indexing for 3D array
    X_test.iloc[student_idx],
    matplotlib=True,
    show=False
)
plt.title(f"Why was Student #{student_idx} flagged as At-Risk?", fontsize=14)
plt.tight_layout()
plt.show()

## 🎯 Conclusion and Real-World Impact

**For Educators:** EduGuard transforms raw data into actionable empathy. Instead of waiting for a student to fail, teachers can see a student is at risk in Week 4 and offer targeted help based on why they are struggling.

**For Administrators:** By understanding the global drivers of dropout, such as previous attempts and heavy course loads, institutions can design better orientation programs and academic advising strategies.

**Technical Achievement:** This project demonstrates the power of combining ensemble machine learning (Random Forest) with modern interpretability techniques (SHAP) to solve a critical real world problem in education.